In [ ]:
from langchain_milvus import Milvus
from langchain_ollama import OllamaEmbeddings

"""
    创建向量化模型
"""
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b", dimensions=1024
)

In [ ]:
from app.core.config import settings
from langchain_milvus import BM25BuiltInFunction

"""
    创建langchain包装过的向量数据库
"""

vector_store = Milvus(
     # 稠密向量模型
    embedding_function=ollama_embeddings,
    # collection名称
    collection_name="langchain_collection",
    # 做稀疏向量的操作函数
    builtin_function=BM25BuiltInFunction(
        # 指定中文分词
        analyzer_params={"type": "chinese"}
    ),
     # 向量字段命名
    vector_field=["dense", "sparse"],
    # 数据库连接地址
    connection_args={
        "uri": settings.rag.milvus_url,
    },
    # 是否删除旧的collection，避免重复创建
    drop_old=False,
    # 自动主键
    auto_id=True
)


In [ ]:
"""
    定义自定义reranker函数
"""
from pymilvus import Function, FunctionType
def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        # 重排函数名
         name="小汪重排ranker",
        input_field_names=["text"],  # 原始文档字段
        function_type=FunctionType.RERANK,  # ranker类型，这里是固定值
        params={
            # 使用模型进行reranker
            "reranker": "model",
            "provider": "ali",  # rerank模型提供者
            "model_name": "gte-rerank-v2",  # rerank模型名称
            "queries": queries,  # 查询条件
            "max_client_batch_size": 5,  # 向模型发送请求时的批处理限制
        },
    )

In [ ]:
user_input = '孔子是谁'

result = vector_store.similarity_search_with_score(
    query=user_input,
    # 最终返回的3条
    k=3,
    # 初筛召回的文档数量
    fetch_k=5,
    # expr=filter
    reranker=create_cross_encoder_ranker([user_input])
)

for document, score in result:
    print(f'文档得分:{score}')
    print(f'文档内容:{document.model_dump_json(indent=2)}')